In [2]:
%pip install mlxtend

  Obtaining dependency information for mlxtend from https://files.pythonhosted.org/packages/61/b8/bda63104fa9e3c4e88bd158b664d21b66667fbee3eb35f737dd329dc26a3/mlxtend-0.25.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
    --------------------------------------- 0.0/1.4 MB 217.9 kB/s eta 0:00:07
   - -------------------------------------- 0.0/1.4 MB 281.8 kB/s eta 0:00:05
   -- ------------------------------------- 0.1/1.4 MB 393.8 kB/s eta 0:00:04
   --- ------------------------------------ 0.1/1.4 MB 554.9 kB/s eta 0:00:03
   -------- ------------------------------- 0.3/1.4 MB 1.1 MB/s eta 0:00:01
   ---------------- ----------------------- 0.6/1.4 MB 1.9 MB/s eta 0:00:01
   -------------------------------- ------- 1.1/1.4 MB 3.2 MB/s eta 0:00:01
   ---------------------------------------  1.4/1.4 M

In [3]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Load the data
df = pd.read_csv('../data/processed/engineered_smartphone_usage.csv')

# 2. Prepare Categorical Data for Association Rules
# Association rules require boolean (True/False) "transactions"
basket = pd.DataFrame()

# We will define "High" as anything above the median average
basket['High_ScreenTime'] = df['Daily_ScreenTime_Hours'] > df['Daily_ScreenTime_Hours'].median()

features = ['SocialMedia_Min', 'Gaming_Min', 'Study_Min', 'Battery_Drain_Percent']
for f in features:
    if f in df.columns:
        # Create a boolean column for "High" usage
        basket[f'High_{f.split("_")[0]}'] = df[f] > df[f].median()

# 3. Apply Apriori Algorithm
# min_support=0.2 means the behavioral combo must happen in at least 20% of the dataset
frequent_itemsets = apriori(basket, min_support=0.2, use_colnames=True)

# 4. Generate Association Rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# Sort by confidence and lift to see the absolute strongest behavioral links
rules = rules.sort_values(by=['confidence', 'lift'], ascending=[False, False])

# 5. Display the top 10 rules
print("--- Top 10 Behavioral Association Rules ---")
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

--- Top 10 Behavioral Association Rules ---


,antecedents,consequents,support,confidence,lift
0,frozenset({High_SocialMedia}),frozenset({High_ScreenTime}),0.2731,0.546856,1.094588
1,frozenset({High_ScreenTime}),frozenset({High_SocialMedia}),0.2731,0.546637,1.094588
2,frozenset({High_Study}),frozenset({High_ScreenTime}),0.2594,0.520779,1.042392
3,frozenset({High_ScreenTime}),frozenset({High_Study}),0.2594,0.519215,1.042392
5,frozenset({High_Gaming}),frozenset({High_SocialMedia}),0.2490,0.502827,1.006863
4,frozenset({High_SocialMedia}),frozenset({High_Gaming}),0.2490,0.498598,1.006863
